# Task 3: Constraint Satisfaction Problems (Kakuro)
**Course:** Introduction to Artificial Intelligence (503043) - Ton Duc Thang University (TDTU)  
**Authors:** Student Group Implementation  

## Table of Contents
1. **CSP Problem Formulation: Variables, Domains, and Run Constraints**
2. **Integrity Validation (Total Horizontal Clues == Total Vertical Clues)**
3. **Solving the Complex 9x9 Benchmark with Backtracking + AC-3 + Heuristics (MRV, LCV)**
4. **Comprehensive Evaluation: Correctness, Stability, and Effectiveness (Ablation Study)**

### 1. Load and Initialize Kakuro CSP from JSON Benchmark

In [1]:
import sys, os
source_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
if os.path.abspath('source') not in sys.path:
    sys.path.insert(0, os.path.abspath('source'))

from task3_kakuro.csp.csp_problem import KakuroCSP

puzzle_path = 'puzzles/complex_9x9.json'
if not os.path.exists(puzzle_path):
    puzzle_path = 'source/task3_kakuro/puzzles/complex_9x9.json'
if not os.path.exists(puzzle_path):
    puzzle_path = 'task3_kakuro/puzzles/complex_9x9.json'

csp = KakuroCSP()
csp.load_from_json(puzzle_path)
print(f'[*] Loaded puzzle benchmark: {csp.name}')
print(f'[*] Board Dimensions: {csp.rows}x{csp.cols}')
print(f'[*] White Cell Variables: {len(csp.variables)}')
print(f'[*] Run Constraints: {len(csp.runs)} (14 horizontal, 14 vertical)')
h_sum, v_sum = csp.get_clue_sums()
print(f'[*] Integrity Check (Total H == Total V): {csp.validate_puzzle_integrity()} (H={h_sum}, V={v_sum})')

[*] Loaded puzzle benchmark: Kakuro Complex 9x9 (Problem Page 5 Test Case)
[*] Board Dimensions: 9x9
[*] White Cell Variables: 48
[*] Run Constraints: 28 (14 horizontal, 14 vertical)
[*] Integrity Check (Total H == Total V): True (H=249, V=249)


### 2. Solve the Complex 9x9 Benchmark & Trace Solver Execution (Solver Tracer)

In [2]:
from task3_kakuro.solver.backtracking_solver import KakuroBacktrackingSolver
from task3_kakuro.solver.tracer import SolverTracer

tracer = SolverTracer()
solver = KakuroBacktrackingSolver(use_ac3=True, use_mrv=True, use_lcv=True, tracer=tracer)

print('[*] Starting Kakuro 9x9 Backtracking Solver...')
result = solver.solve(csp)

print('-' * 40)
print(' SOLVE RESULT')
print('-' * 40)
print(f' - Solved successfully   : {result.is_solved}')
print(f' - Execution time        : {result.execution_time:.4f} seconds')
print(f' - Explored nodes        : {result.nodes_explored}')
print(f' - Backtracks count      : {result.backtracks}')
print(f' - Tracer events logged  : {len(tracer.events)}')

if result.is_solved:
    print('\n[*] Solution Grid:')
    for r in range(csp.rows):
        row_str = []
        for c in range(csp.cols):
            pos = (r, c)
            if pos in csp.variables:
                row_str.append(f' {result.solution_grid[pos]} ')
            else:
                row_str.append(' # ')
        print('  ' + ''.join(row_str))
    print('\n[+] All clue constraints satisfied 100%!')

[*] Starting Kakuro 9x9 Backtracking Solver...
----------------------------------------
 SOLVE RESULT
----------------------------------------
 - Solved successfully   : True
 - Execution time        : 0.2757 seconds
 - Explored nodes        : 11383
 - Backtracks count      : 11335
 - Tracer events logged  : 31246

[*] Solution Grid:
   #  #  #  #  #  #  #  #  # 
   #  #  5  9  8  #  1  2  # 
   #  #  2  7  1  3  4  6  5 
   #  8  6  #  #  1  2  8  9 
   #  9  8  #  3  2  8  9  # 
   #  #  9  8  6  5  #  7  5 
   #  3  4  5  2  #  #  4  2 
   #  9  3  7  1  6  8  5  # 
   #  #  7  9  #  1  4  3  # 

[+] All clue constraints satisfied 100%!


### 3. Comprehensive Evaluation: Correctness, Stability, and Effectiveness (Ablation Study)

In [3]:
from task3_kakuro.experiments.evaluate_kakuro import KakuroEvaluator

evaluator = KakuroEvaluator(puzzle_path=puzzle_path)
evaluator.run_all_evaluations()

     KAKURO CSP SOLVER PERFORMANCE & STABILITY EVALUATION

[*] Stability Test across 5 independent runs:
  - Run 1: Time=0.2173s, Nodes=11383, Backtracks=11335
  - Run 2: Time=0.2816s, Nodes=11383, Backtracks=11335
  - Run 3: Time=0.1936s, Nodes=11383, Backtracks=11335
  - Run 4: Time=0.2010s, Nodes=11383, Backtracks=11335
  - Run 5: Time=0.1973s, Nodes=11383, Backtracks=11335
[+] Correctness (Independent Verification): VALID 100%
[+] Solution Consistency: 100% IDENTICAL
[+] Average Time: 0.2182 seconds

[*] Ablation Study - Comparing Heuristic Configurations:
  * Pure Backtracking (No Heuristics, Arbitrary): Timeout (>3.0s) | Nodes: >5,502 | Backtracks: >5,502
  * Backtracking + MRV: 0.5545s | Nodes: 35,075 | Backtracks: 35,027
  * Backtracking + MRV + LCV: 0.1748s | Nodes: 11,383 | Backtracks: 11,335
  * Backtracking + AC-3 + MRV + LCV (Full): 0.1885s | Nodes: 11,383 | Backtracks: 11,335
